# 05 - Longstay Careunit Burden

Author: Saige Mukherjee

Contact: mukherjeesaige@gmail.com //
https://www.linkedin.com/in/saige-mukherjee-0aba68281/

The analysis runs inside the Jupyter notebook. Run the notebook with credentialed access to generate the results.

If you want to run the notebook and generate the report:
- obtained credentialed access to MIMIC-IV via PhysioNet and BigQuery,
- create a GCP project and star the MIMIC-IV dataset ,
- enter the GCP project ID below and execute the program.

In [ ]:
PROJECT_ID = ""  # Enter your Google Cloud billing project ID before running.

## Main measures

- **Long segment:** duration greater than a selected threshold.
- **Primary threshold:** 72 hours.
- **Hours in long segments:** the full duration of segments above the threshold.
- **Excess hours beyond 72 hours:** `max(segment_hours - 72, 0)`, summed across segments.
- **Concentration:** share and cumulative share of all excess hours attributable to each care unit.

## Privacy and publication safeguards

- No patient-, admission-, or transfer-level rows are displayed.
- Queries return aggregate results only.
- Grouped outputs require at least 11 long segments and 11 distinct patients unless the count is zero.
- No MIMIC-derived CSV, Parquet, spreadsheet, database, or pickle files are written.
- `PROJECT_ID` must remain empty before public sharing.


In [ ]:
# Uncomment in a fresh environment if required.
# %pip install -q google-cloud-bigquery db-dtypes pandas numpy matplotlib


## 0. Environment and configuration


In [ ]:
from __future__ import annotations

import warnings
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from google.cloud import bigquery

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})


In [ ]:
HOSP_DATASET = "physionet-data.mimiciv_3_1_hosp"
TRANSFERS_TABLE = f"`{HOSP_DATASET}.transfers`"

INCLUDE_ED_ONLY = True
PRIMARY_THRESHOLD_HOURS = 72
THRESHOLDS_HOURS = [48, 72, 168, 336]
MAX_SEGMENT_HOURS = 365 * 24
MIN_CELL_N = 11
TOP_N_CAREUNITS = 20
MAX_BYTES_BILLED = 5_000_000_000

# Stable MIMIC-IV v3.1 validation targets for this transfer cohort.
EXPECTED_TOTAL_VALID_SEGMENTS = 1_867_366
EXPECTED_ED_ONLY_SEGMENTS = 408_882
EXPECTED_ADMISSION_LINKED_SEGMENTS = 1_458_484
EXPECTED_REPRESENTED_ADMISSIONS = 545_994

if not PROJECT_ID.strip():
    raise ValueError("Set PROJECT_ID before running the notebook.")

if INCLUDE_ED_ONLY is not True:
    raise ValueError("This notebook is designed to include ED-only segments.")

client = bigquery.Client(project=PROJECT_ID)

print(f"Transfers table: {TRANSFERS_TABLE}")
print(f"Include ED-only rows with NULL hadm_id: {INCLUDE_ED_ONLY}")
print(f"Primary threshold: {PRIMARY_THRESHOLD_HOURS} hours")

## 1. Helper functions

The display guard suppresses an aggregate row if any count-like value is between 1 and 10. SQL queries also enforce minimum long-segment and distinct-patient counts for grouped outputs.


In [ ]:
def run_query(sql: str, job_label: str | None = None) -> pd.DataFrame:
    """Run an aggregate BigQuery query and return a pandas DataFrame."""
    job_config = bigquery.QueryJobConfig(
        maximum_bytes_billed=MAX_BYTES_BILLED
    )
    if job_label:
        job_config.labels = {
            "notebook": "05-longstay-careunit-burden",
            "step": job_label[:63].lower().replace("_", "-"),
        }

    return client.query(sql, job_config=job_config).to_dataframe(
        create_bqstorage_client=False
    )


def likely_count_columns(df: pd.DataFrame) -> list[str]:
    """Identify columns that likely contain aggregate counts."""
    exact = {
        "n", "count", "segments", "patients", "admissions",
        "segment_n", "patient_n", "admission_n",
        "long_segment_n", "long_patient_n", "long_admission_n",
        "valid_segments", "ed_only_segments",
        "admission_linked_segments", "represented_admissions",
        "excluded_over_365d_n",
    }

    found: list[str] = []
    for col in df.columns:
        name = col.lower()
        count_like = (
            name in exact
            or name.startswith("n_")
            or name.endswith("_n")
            or name.endswith("_count")
            or name.endswith("_segments")
            or name.endswith("_patients")
            or name.endswith("_admissions")
        )
        if count_like and pd.api.types.is_numeric_dtype(df[col]):
            found.append(col)
    return found


def safe_display(
    df: pd.DataFrame,
    max_rows: int = 30,
    extra_count_columns: Iterable[str] = (),
) -> None:
    """Suppress rows containing any count from 1 through MIN_CELL_N - 1."""
    out = df.copy()
    count_cols = list(dict.fromkeys(
        likely_count_columns(out) + list(extra_count_columns)
    ))

    small_mask = pd.Series(False, index=out.index)
    for col in count_cols:
        values = pd.to_numeric(out[col], errors="coerce")
        small_mask |= values.between(1, MIN_CELL_N - 1, inclusive="both")

    suppressed_n = int(small_mask.sum())
    out = out.loc[~small_mask].copy()

    display(out.head(max_rows))
    if suppressed_n:
        print(f"Suppressed {suppressed_n:,} aggregate row(s) with a count from 1 to 10.")
    if len(out) > max_rows:
        print(f"Showing first {max_rows:,} of {len(out):,} publishable aggregate rows.")
    return None


def format_hours(value: float) -> str:
    """Readable large-hour formatter."""
    if pd.isna(value):
        return "NA"
    if abs(value) >= 1_000_000:
        return f"{value / 1_000_000:,.2f} million hours"
    if abs(value) >= 1_000:
        return f"{value / 1_000:,.1f} thousand hours"
    return f"{value:,.1f} hours"


def first_rank_reaching(df: pd.DataFrame, cumulative_col: str, target: float) -> int | None:
    """Return the first rank reaching a cumulative percentage target."""
    reached = df.loc[df[cumulative_col] >= target, "rank"]
    return int(reached.iloc[0]) if not reached.empty else None


## 2. Broad operational care-unit groups

MIMIC care-unit labels are administrative and operational labels rather than a perfect ward taxonomy. The mapping below supports broad comparisons while preserving the granular care-unit analysis.


In [ ]:
CAREUNIT_TO_BROAD_GROUP = {
    "Emergency Department": "Emergency / Observation",
    "Emergency Department Observation": "Emergency / Observation",
    "Observation": "Emergency / Observation",
    "Discharge Lounge": "Discharge / Transition",

    "Medicine": "Medical Ward",
    "Med/Surg": "Medical Ward",
    "Medical/Surgical": "Medical Ward",
    "Pulmonary": "Medical Ward",
    "Infectious Disease": "Medical Ward",
    "Geriatrics": "Medical Ward",

    "Medicine/Cardiology": "Cardiology",
    "Medicine/Cardiology Intermediate": "Cardiology",
    "Cardiology": "Cardiology",
    "Cardiology Surgery Intermediate": "Cardiology",

    "Hematology/Oncology": "Oncology / Hematology",
    "Hematology/Oncology Intermediate": "Oncology / Hematology",
    "Oncology": "Oncology / Hematology",
    "Transplant": "Transplant",

    "Surgery": "Surgical Ward",
    "Surgery/Trauma": "Surgical Ward",
    "Med/Surg/Trauma": "Surgical Ward",
    "Vascular": "Surgical Ward",
    "Thoracic Surgery": "Surgical Ward",
    "Plastic Surgery": "Surgical Ward",
    "Urology": "Surgical Ward",
    "Orthopaedics": "Surgical Ward",
    "Gynecology": "Surgical Ward",

    "Obstetrics": "Obstetrics / Gynecology",
    "Labor & Delivery": "Obstetrics / Gynecology",

    "Operating Room": "Procedural / OR",
    "PACU": "Procedural / OR",
    "Post-Anesthesia Care Unit": "Procedural / OR",

    "Neurology": "Neurology",
    "Neurology Intermediate": "Neurology",
    "Neuro Intermediate": "Neurology",
    "Neuro Stepdown": "Neurology",

    "Psychiatry": "Psychiatry",

    "Medical Intensive Care Unit (MICU)": "ICU",
    "Surgical Intensive Care Unit (SICU)": "ICU",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)": "ICU",
    "Cardiac Vascular Intensive Care Unit (CVICU)": "ICU",
    "Coronary Care Unit (CCU)": "ICU",
    "Trauma SICU (TSICU)": "ICU",
    "Neuro Surgical Intensive Care Unit (Neuro SICU)": "ICU",
    "Intensive Care Unit": "ICU",

    "Intermediate": "Intermediate Care",
    "Medical Intermediate": "Intermediate Care",
    "Surgical Intermediate": "Intermediate Care",

    "Pediatrics": "Pediatrics",
    "Special Care Nursery": "Pediatrics / Neonatal",
    "Nursery": "Pediatrics / Neonatal",
}


def sql_string_literal(value: str) -> str:
    return "'" + value.replace("'", "\\'") + "'"


def build_broad_group_case(
    mapping: dict[str, str],
    source_col: str = "careunit",
) -> str:
    lines = ["CASE"]
    for careunit, group in sorted(mapping.items()):
        lines.append(
            f"  WHEN {source_col} = {sql_string_literal(careunit)} "
            f"THEN {sql_string_literal(group)}"
        )
    lines.append("  ELSE 'Other / Unmapped'")
    lines.append("END")
    return "\n".join(lines)


BROAD_GROUP_CASE = build_broad_group_case(CAREUNIT_TO_BROAD_GROUP)


## 3. Shared transfer cohort

This is the sole system-wide cohort definition used throughout the notebook.

`hadm_id` is retained only as an aggregate linkage indicator. It is **not** an inclusion requirement.


In [ ]:
scope_filter_sql = "TRUE" if INCLUDE_ED_ONLY else "hadm_id IS NOT NULL"

TRANSFER_COHORT_CTE = f"""
WITH transfer_candidates AS (
    SELECT
        subject_id,
        hadm_id,
        eventtype,
        careunit,
        DATETIME_DIFF(outtime, intime, SECOND) / 3600.0
            AS hours_in_careunit
    FROM {TRANSFERS_TABLE}
    WHERE careunit IS NOT NULL
      AND intime IS NOT NULL
      AND outtime IS NOT NULL
      AND outtime > intime
      AND ({scope_filter_sql})
),
transfer_cohort AS (
    SELECT
        subject_id,
        hadm_id,
        eventtype,
        careunit,
        hours_in_careunit
    FROM transfer_candidates
    WHERE hours_in_careunit > 0
      AND hours_in_careunit <= {MAX_SEGMENT_HOURS}
)
"""

assert INCLUDE_ED_ONLY is True
assert "hadm_id IS NOT NULL" not in TRANSFER_COHORT_CTE
assert "JOIN" not in TRANSFER_COHORT_CTE.upper()

print("Shared cohort is transfer-based and includes NULL hadm_id rows.")


## 4. Cohort audit and ED-inclusion proof

For MIMIC-IV v3.1, the corrected cohort should contain:

- **1,867,366** valid segments;
- **408,882** ED-only segments;
- **1,458,484** admission-linked segments;
- **545,994** represented hospital admissions;
- **<11** positive-duration segments above 365 days excluded.

The assertions prevent an accidental return to an admission-only cohort.


In [ ]:
sql_scope_audit = f"""
{TRANSFER_COHORT_CTE}
SELECT
    COUNT(*) AS valid_segments,
    COUNT(DISTINCT subject_id) AS unique_patients,
    COUNT(DISTINCT hadm_id) AS represented_admissions,
    COUNTIF(hadm_id IS NULL) AS ed_only_segments,
    COUNTIF(hadm_id IS NOT NULL) AS admission_linked_segments,
    ROUND(
        100 * SAFE_DIVIDE(COUNTIF(hadm_id IS NULL), COUNT(*)),
        2
    ) AS pct_segments_ed_only,
    (
        SELECT COUNT(*) > 0 AND COUNT(*) < {MIN_CELL_N}
        FROM transfer_candidates
        WHERE hours_in_careunit > {MAX_SEGMENT_HOURS}
    ) AS excluded_over_365d_is_small_cell
FROM transfer_cohort
"""

scope_audit_df = run_query(sql_scope_audit, "scope-audit")
observed = scope_audit_df.iloc[0]

def mask_small_n(val):
    if pd.isna(val): return val
    if isinstance(val, (bool, np.bool_)): return val
    try:
        num = float(val)
        return "<11" if 0 < num < MIN_CELL_N else f"{int(num):,}"
    except (ValueError, TypeError):
        return val

display_audit = scope_audit_df.T.rename(columns={0: "value"})
display_audit["value"] = display_audit["value"].apply(mask_small_n)
display(display_audit)

checks = pd.DataFrame({
    "check": [
        "All valid segments",
        "ED-only segments included",
        "Admission-linked segments",
        "Represented admissions",
    ],
    "observed_raw": [
        int(observed["valid_segments"]),
        int(observed["ed_only_segments"]),
        int(observed["admission_linked_segments"]),
        int(observed["represented_admissions"]),
    ],
    "expected_raw": [
        EXPECTED_TOTAL_VALID_SEGMENTS,
        EXPECTED_ED_ONLY_SEGMENTS,
        EXPECTED_ADMISSION_LINKED_SEGMENTS,
        EXPECTED_REPRESENTED_ADMISSIONS,
    ],
})

checks["matches_expected"] = (
    checks["observed_raw"] == checks["expected_raw"]
)

checks_display = checks.copy()
checks_display["observed"] = checks_display["observed_raw"].apply(mask_small_n)
checks_display["expected"] = checks_display["expected_raw"].apply(mask_small_n)
display(checks_display[["check", "observed", "expected", "matches_expected"]])

assert int(observed["ed_only_segments"]) > 0, "ED-only rows were not included."
assert int(observed["valid_segments"]) == (
    int(observed["ed_only_segments"])
    + int(observed["admission_linked_segments"])
)
assert bool(observed["excluded_over_365d_is_small_cell"]), (
    "The excluded over-cap group is not within the publication threshold."
)
assert checks["matches_expected"].all(), (
    "The cohort does not match the stored MIMIC-IV v3.1 validation targets."
)

print("Cohort validation passed: ED-only segments are included.")

## 5. Primary 72-hour system-wide burden

A long segment lasts more than 72 hours. Only time after the first 72 hours contributes to the excess-hours measure.


In [ ]:
sql_primary_burden = f"""
{TRANSFER_COHORT_CTE}
SELECT
    COUNT(*) AS segment_n,
    COUNT(DISTINCT subject_id) AS patient_n,
    COUNT(DISTINCT hadm_id) AS admission_n,

    COUNTIF(hadm_id IS NULL) AS ed_only_segment_n,
    COUNTIF(hadm_id IS NOT NULL) AS admission_linked_segment_n,

    COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS})
        AS long_segment_n,
    COUNT(DISTINCT IF(
        hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
        subject_id,
        NULL
    )) AS long_patient_n,
    COUNT(DISTINCT IF(
        hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
        hadm_id,
        NULL
    )) AS long_admission_n,

    ROUND(
        100 * SAFE_DIVIDE(
            COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS}),
            COUNT(*)
        ),
        2
    ) AS pct_segments_long,

    ROUND(SUM(hours_in_careunit), 2) AS all_segment_hours,
    ROUND(SUM(IF(
        hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
        hours_in_careunit,
        0
    )), 2) AS hours_in_long_segments,
    ROUND(
        100 * SAFE_DIVIDE(
            SUM(IF(
                hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
                hours_in_careunit,
                0
            )),
            SUM(hours_in_careunit)
        ),
        2
    ) AS pct_total_hours_in_long_segments,

    ROUND(SUM(GREATEST(
        hours_in_careunit - {PRIMARY_THRESHOLD_HOURS},
        0
    )), 2) AS excess_hours_beyond_72h,

    ROUND(
        APPROX_QUANTILES(
            IF(
                hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
                hours_in_careunit,
                NULL
            ),
            100
        )[OFFSET(50)],
        2
    ) AS median_long_segment_hours,
    ROUND(
        APPROX_QUANTILES(
            IF(
                hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
                hours_in_careunit,
                NULL
            ),
            100
        )[OFFSET(90)],
        2
    ) AS p90_long_segment_hours
FROM transfer_cohort
"""

primary_burden_df = run_query(sql_primary_burden, "primary-burden")
display(primary_burden_df.T.rename(columns={0: "value"}))


## 6. Threshold sensitivity

This checks whether the long-tail conclusion is specific to the 72-hour threshold or remains visible at 48 hours, 7 days, and 14 days.


In [ ]:
threshold_sql_values = ", ".join(str(x) for x in THRESHOLDS_HOURS)

sql_threshold_sensitivity = f"""
{TRANSFER_COHORT_CTE},
thresholds AS (
    SELECT threshold_hours
    FROM UNNEST([{threshold_sql_values}]) AS threshold_hours
),
totals AS (
    SELECT
        COUNT(*) AS all_segments,
        SUM(hours_in_careunit) AS all_segment_hours
    FROM transfer_cohort
),
threshold_rollup AS (
    SELECT
        threshold_hours,
        COUNTIF(hours_in_careunit > threshold_hours)
            AS long_segment_n,
        COUNT(DISTINCT IF(
            hours_in_careunit > threshold_hours,
            subject_id,
            NULL
        )) AS long_patient_n,
        COUNT(DISTINCT IF(
            hours_in_careunit > threshold_hours,
            hadm_id,
            NULL
        )) AS long_admission_n,
        SUM(IF(
            hours_in_careunit > threshold_hours,
            hours_in_careunit,
            0
        )) AS hours_in_long_segments,
        SUM(GREATEST(hours_in_careunit - threshold_hours, 0))
            AS excess_hours
    FROM transfer_cohort
    CROSS JOIN thresholds
    GROUP BY threshold_hours
)
SELECT
    threshold_hours,
    long_segment_n,
    long_patient_n,
    long_admission_n,
    ROUND(
        100 * SAFE_DIVIDE(long_segment_n, totals.all_segments),
        2
    ) AS pct_segments_long,
    ROUND(hours_in_long_segments, 2) AS hours_in_long_segments,
    ROUND(
        100 * SAFE_DIVIDE(
            hours_in_long_segments,
            totals.all_segment_hours
        ),
        2
    ) AS pct_total_hours_in_long_segments,
    ROUND(excess_hours, 2) AS excess_hours
FROM threshold_rollup
CROSS JOIN totals
WHERE long_segment_n = 0
   OR (
       long_segment_n >= {MIN_CELL_N}
       AND long_patient_n >= {MIN_CELL_N}
   )
ORDER BY threshold_hours
"""

threshold_sensitivity_df = run_query(
    sql_threshold_sensitivity,
    "threshold-sensitivity",
)
safe_display(threshold_sensitivity_df, max_rows=20)


In [ ]:
plot_df = threshold_sensitivity_df.copy()
plot_df["threshold_label"] = plot_df["threshold_hours"].map({
    48: "48 h",
    72: "72 h",
    168: "7 d",
    336: "14 d",
})

x = np.arange(len(plot_df))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 5.5))
bars1 = ax.bar(
    x - width / 2,
    plot_df["pct_segments_long"],
    width,
    label="Share of segments",
)
bars2 = ax.bar(
    x + width / 2,
    plot_df["pct_total_hours_in_long_segments"],
    width,
    label="Share of segment-hours",
)

ax.set_xticks(x, plot_df["threshold_label"])
ax.set_ylabel("Percent")
ax.set_title("Long segments consume a disproportionate share of recorded time")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)

ax.bar_label(
    bars1,
    labels=[f"{v:.1f}%" for v in plot_df["pct_segments_long"]],
    padding=3,
    fontsize=9,
)
ax.bar_label(
    bars2,
    labels=[
        f"{v:.1f}%"
        for v in plot_df["pct_total_hours_in_long_segments"]
    ],
    padding=3,
    fontsize=9,
)

plt.tight_layout()
plt.show()


## 7. Care-unit leaderboard at 72 hours

The leaderboard ranks granular care units by total hours accumulated after the first 72 hours of each segment.

A high rank identifies where long-duration burden is concentrated. It does **not** establish inefficiency or avoidable delay.


In [ ]:
sql_careunit_burden = f"""
{TRANSFER_COHORT_CTE},
totals AS (
    SELECT
        SUM(GREATEST(
            hours_in_careunit - {PRIMARY_THRESHOLD_HOURS},
            0
        )) AS all_excess_hours
    FROM transfer_cohort
),
careunit_rollup AS (
    SELECT
        careunit,
        COUNT(*) AS segment_n,
        COUNT(DISTINCT subject_id) AS patient_n,
        COUNT(DISTINCT hadm_id) AS admission_n,
        COUNTIF(hadm_id IS NULL) AS ed_only_segment_n,

        COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS})
            AS long_segment_n,
        COUNT(DISTINCT IF(
            hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
            subject_id,
            NULL
        )) AS long_patient_n,
        COUNT(DISTINCT IF(
            hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
            hadm_id,
            NULL
        )) AS long_admission_n,
        COUNTIF(
            hours_in_careunit > {PRIMARY_THRESHOLD_HOURS}
            AND hadm_id IS NULL
        ) AS ed_only_long_segment_n,

        ROUND(
            100 * SAFE_DIVIDE(
                COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS}),
                COUNT(*)
            ),
            2
        ) AS pct_segments_long,

        ROUND(SUM(IF(
            hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
            hours_in_careunit,
            0
        )), 2) AS hours_in_long_segments,

        ROUND(SUM(GREATEST(
            hours_in_careunit - {PRIMARY_THRESHOLD_HOURS},
            0
        )), 2) AS excess_hours_beyond_72h,

        ROUND(AVG(IF(
            hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
            hours_in_careunit - {PRIMARY_THRESHOLD_HOURS},
            NULL
        )), 2) AS mean_excess_hours_per_long_segment,

        ROUND(
            APPROX_QUANTILES(
                IF(
                    hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
                    hours_in_careunit,
                    NULL
                ),
                100
            )[OFFSET(50)],
            2
        ) AS median_long_segment_hours,

        ROUND(
            APPROX_QUANTILES(
                IF(
                    hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
                    hours_in_careunit,
                    NULL
                ),
                100
            )[OFFSET(90)],
            2
        ) AS p90_long_segment_hours
    FROM transfer_cohort
    GROUP BY careunit
)
SELECT
    r.*,
    ROUND(
        100 * SAFE_DIVIDE(
            r.excess_hours_beyond_72h,
            totals.all_excess_hours
        ),
        2
    ) AS pct_of_all_excess_hours
FROM careunit_rollup AS r
CROSS JOIN totals
WHERE r.long_segment_n >= {MIN_CELL_N}
  AND r.long_patient_n >= {MIN_CELL_N}
ORDER BY r.excess_hours_beyond_72h DESC
"""

careunit_burden_df = run_query(sql_careunit_burden, "careunit-burden")

careunit_burden_df = (
    careunit_burden_df
    .sort_values("excess_hours_beyond_72h", ascending=False)
    .reset_index(drop=True)
)
careunit_burden_df["rank"] = np.arange(1, len(careunit_burden_df) + 1)
careunit_burden_df["cumulative_pct_of_all_excess_hours"] = (
    careunit_burden_df["pct_of_all_excess_hours"].cumsum()
)

careunit_display_cols = [
    "rank",
    "careunit",
    "segment_n",
    "patient_n",
    "ed_only_segment_n",
    "long_segment_n",
    "ed_only_long_segment_n",
    "pct_segments_long",
    "median_long_segment_hours",
    "p90_long_segment_hours",
    "mean_excess_hours_per_long_segment",
    "excess_hours_beyond_72h",
    "pct_of_all_excess_hours",
    "cumulative_pct_of_all_excess_hours",
]

publishable_careunit_df = safe_display(
    careunit_burden_df[careunit_display_cols],
    max_rows=30,
)


In [ ]:
top_units_plot_df = (
    careunit_burden_df
    .head(TOP_N_CAREUNITS)
    .sort_values("excess_hours_beyond_72h")
    .copy()
)

# Increased figure width from 11 to 14
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(
    top_units_plot_df["careunit"],
    top_units_plot_df["excess_hours_beyond_72h"] / 1_000_000, # Changed to millions
)

ax.set_xlabel("Excess recorded care-unit hours beyond 72 h (millions)")
ax.set_ylabel("Care unit")
ax.set_title("Care units with the largest long-duration burden")
ax.grid(axis="x", alpha=0.25)

ax.bar_label(
    bars,
    labels=[
        f"{value / 1_000_000:,.2f}M"
        for value in top_units_plot_df["excess_hours_beyond_72h"]
    ],
    padding=3,
    fontsize=8,
)

plt.tight_layout()
plt.show()

### Pareto concentration of excess hours


In [ ]:
pareto_df = careunit_burden_df.copy()

# Increased figure width from 11 to 14
fig, ax1 = plt.subplots(figsize=(14, 5.8))
x = np.arange(1, len(pareto_df) + 1)

ax1.bar(
    x,
    pareto_df["excess_hours_beyond_72h"] / 1_000_000,
)
ax1.set_xlabel("Care-unit rank")
ax1.set_ylabel("Excess hours beyond 72 h (millions)")
ax1.set_title("Excess long-duration burden is concentrated across care units")
ax1.grid(axis="y", alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(
    x,
    pareto_df["cumulative_pct_of_all_excess_hours"],
    marker="o",
    markersize=3,
    color="black", # Changed line color to black
)
ax2.set_ylabel("Cumulative share of all excess hours (%)")
ax2.set_ylim(0, 105)
ax2.axhline(80, linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()

concentration_summary_df = pd.DataFrame({
    "cumulative_target_pct": [50, 80, 90],
    "careunits_needed": [
        first_rank_reaching(
            pareto_df,
            "cumulative_pct_of_all_excess_hours",
            target,
        )
        for target in [50, 80, 90]
    ],
})
display(concentration_summary_df)

## 8. Burden versus intensity

The quadrant plot separates two different operational patterns:

- **volume:** many long segments;
- **intensity:** more excess hours per long segment.

A care unit can rank highly because it has many moderately long segments, fewer extremely long segments, or both.


In [ ]:
quadrant_df = careunit_burden_df.loc[
    (careunit_burden_df["long_segment_n"] > 0)
    & (careunit_burden_df["mean_excess_hours_per_long_segment"] > 0)
].copy()

x_mid = quadrant_df["long_segment_n"].median()
y_mid = quadrant_df["mean_excess_hours_per_long_segment"].median()

fig, ax = plt.subplots(figsize=(11, 6.5))
sizes = (
    40
    + 500
    * quadrant_df["pct_of_all_excess_hours"]
    / quadrant_df["pct_of_all_excess_hours"].max()
)

ax.scatter(
    quadrant_df["long_segment_n"],
    quadrant_df["mean_excess_hours_per_long_segment"],
    s=sizes,
    alpha=0.7,
)

ax.set_xscale("log")
ax.axvline(x_mid, linestyle="--", linewidth=1)
ax.axhline(y_mid, linestyle="--", linewidth=1)
ax.set_xlabel("Long segments above 72 h (log scale)")
ax.set_ylabel("Mean excess hours per long segment")
ax.set_title("Long-stay volume and intensity are distinct")

for _, row in quadrant_df.head(12).iterrows():
    ax.annotate(
        row["careunit"],
        (
            row["long_segment_n"],
            row["mean_excess_hours_per_long_segment"],
        ),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
    )

ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 9. Broad operational-group burden

This aggregates granular care-unit labels into broader operational groups. The granular leaderboard remains the primary result.


In [ ]:
sql_broad_group_burden = f"""
{TRANSFER_COHORT_CTE},
labeled AS (
    SELECT
        *,
        {BROAD_GROUP_CASE} AS broad_careunit_group
    FROM transfer_cohort
),
totals AS (
    SELECT
        SUM(GREATEST(
            hours_in_careunit - {PRIMARY_THRESHOLD_HOURS},
            0
        )) AS all_excess_hours
    FROM labeled
),
broad_group_rollup AS (
    SELECT
        broad_careunit_group,
        COUNT(*) AS segment_n,
        COUNT(DISTINCT subject_id) AS patient_n,
        COUNT(DISTINCT hadm_id) AS admission_n,
        COUNTIF(hadm_id IS NULL) AS ed_only_segment_n,

        COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS})
            AS long_segment_n,
        COUNT(DISTINCT IF(
            hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
            subject_id,
            NULL
        )) AS long_patient_n,
        COUNTIF(
            hours_in_careunit > {PRIMARY_THRESHOLD_HOURS}
            AND hadm_id IS NULL
        ) AS ed_only_long_segment_n,

        ROUND(
            100 * SAFE_DIVIDE(
                COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS}),
                COUNT(*)
            ),
            2
        ) AS pct_segments_long,

        ROUND(SUM(GREATEST(
            hours_in_careunit - {PRIMARY_THRESHOLD_HOURS},
            0
        )), 2) AS excess_hours_beyond_72h,

        ROUND(AVG(IF(
            hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
            hours_in_careunit - {PRIMARY_THRESHOLD_HOURS},
            NULL
        )), 2) AS mean_excess_hours_per_long_segment,

        ROUND(
            APPROX_QUANTILES(
                IF(
                    hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
                    hours_in_careunit,
                    NULL
                ),
                100
            )[OFFSET(50)],
            2
        ) AS median_long_segment_hours
    FROM labeled
    GROUP BY broad_careunit_group
)
SELECT
    r.*,
    ROUND(
        100 * SAFE_DIVIDE(
            r.excess_hours_beyond_72h,
            totals.all_excess_hours
        ),
        2
    ) AS pct_of_all_excess_hours
FROM broad_group_rollup AS r
CROSS JOIN totals
WHERE r.long_segment_n >= {MIN_CELL_N}
  AND r.long_patient_n >= {MIN_CELL_N}
ORDER BY r.excess_hours_beyond_72h DESC
"""

broad_group_burden_df = run_query(
    sql_broad_group_burden,
    "broad-group-burden",
)
safe_display(broad_group_burden_df, max_rows=30)


In [ ]:
broad_plot_df = (
    broad_group_burden_df
    .sort_values("excess_hours_beyond_72h")
    .copy()
)

fig, ax = plt.subplots(figsize=(10, 6.5))
bars = ax.barh(
    broad_plot_df["broad_careunit_group"],
    broad_plot_df["excess_hours_beyond_72h"] / 1_000_000,
)
ax.set_xlabel("Excess recorded care-unit hours beyond 72 h (millions)")
ax.set_ylabel("Broad operational group")
ax.set_title("Broad-group contribution to long-duration burden")
ax.grid(axis="x", alpha=0.25)

ax.bar_label(
    bars,
    labels=[
        f"{value / 1_000_000:,.2f}M"
        for value in broad_plot_df["excess_hours_beyond_72h"]
    ],
    padding=3,
    fontsize=8,
)

plt.tight_layout()
plt.show()


### Unmapped care-unit audit


In [ ]:
sql_unmapped_audit = f"""
{TRANSFER_COHORT_CTE},
labeled AS (
    SELECT
        *,
        {BROAD_GROUP_CASE} AS broad_careunit_group
    FROM transfer_cohort
)
SELECT
    careunit,
    COUNT(*) AS segment_n,
    COUNT(DISTINCT subject_id) AS patient_n,
    COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS})
        AS long_segment_n,
    ROUND(SUM(GREATEST(
        hours_in_careunit - {PRIMARY_THRESHOLD_HOURS},
        0
    )), 2) AS excess_hours_beyond_72h
FROM labeled
WHERE broad_careunit_group = 'Other / Unmapped'
GROUP BY careunit
HAVING segment_n >= {MIN_CELL_N}
   AND patient_n >= {MIN_CELL_N}
ORDER BY excess_hours_beyond_72h DESC
"""

unmapped_audit_df = run_query(sql_unmapped_audit, "unmapped-audit")
safe_display(unmapped_audit_df, max_rows=50)


## 10. ED-only versus admission-linked burden

This section makes the cohort correction visible rather than merely relying on a null-inclusive query.

The ED-only group consists of valid transfer rows with `hadm_id IS NULL`. Admission-linked rows have a non-null `hadm_id`.


In [ ]:
sql_cohort_component_burden = f"""
{TRANSFER_COHORT_CTE}
SELECT
    CASE
        WHEN hadm_id IS NULL THEN 'ED-only'
        ELSE 'Admission-linked'
    END AS cohort_component,

    COUNT(*) AS segment_n,
    COUNT(DISTINCT subject_id) AS patient_n,
    COUNT(DISTINCT hadm_id) AS admission_n,

    COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS})
        AS long_segment_n,
    COUNT(DISTINCT IF(
        hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
        subject_id,
        NULL
    )) AS long_patient_n,

    ROUND(
        100 * SAFE_DIVIDE(
            COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS}),
            COUNT(*)
        ),
        2
    ) AS pct_segments_long,

    ROUND(SUM(hours_in_careunit), 2) AS total_segment_hours,
    ROUND(SUM(IF(
        hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
        hours_in_careunit,
        0
    )), 2) AS hours_in_long_segments,
    ROUND(SUM(GREATEST(
        hours_in_careunit - {PRIMARY_THRESHOLD_HOURS},
        0
    )), 2) AS excess_hours_beyond_72h,

    ROUND(
        APPROX_QUANTILES(hours_in_careunit, 100)[OFFSET(50)],
        2
    ) AS median_segment_hours,
    ROUND(
        APPROX_QUANTILES(
            IF(
                hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
                hours_in_careunit,
                NULL
            ),
            100
        )[OFFSET(50)],
        2
    ) AS median_long_segment_hours
FROM transfer_cohort
GROUP BY cohort_component
ORDER BY cohort_component
"""

cohort_component_burden_df = run_query(
    sql_cohort_component_burden,
    "cohort-component-burden",
)
safe_display(cohort_component_burden_df, max_rows=10)


### ED-only long-duration burden by care unit


In [ ]:
sql_ed_only_careunit_burden = f"""
{TRANSFER_COHORT_CTE}
SELECT
    careunit,
    COUNT(*) AS segment_n,
    COUNT(DISTINCT subject_id) AS patient_n,

    COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS})
        AS long_segment_n,
    COUNT(DISTINCT IF(
        hours_in_careunit > {PRIMARY_THRESHOLD_HOURS},
        subject_id,
        NULL
    )) AS long_patient_n,

    ROUND(
        100 * SAFE_DIVIDE(
            COUNTIF(hours_in_careunit > {PRIMARY_THRESHOLD_HOURS}),
            COUNT(*)
        ),
        2
    ) AS pct_segments_long,

    ROUND(SUM(GREATEST(
        hours_in_careunit - {PRIMARY_THRESHOLD_HOURS},
        0
    )), 2) AS excess_hours_beyond_72h
FROM transfer_cohort
WHERE hadm_id IS NULL
GROUP BY careunit
HAVING long_segment_n >= {MIN_CELL_N}
   AND long_patient_n >= {MIN_CELL_N}
ORDER BY excess_hours_beyond_72h DESC
"""

ed_only_careunit_burden_df = run_query(
    sql_ed_only_careunit_burden,
    "ed-only-careunit-burden",
)
safe_display(ed_only_careunit_burden_df, max_rows=30)


## 11. Care-unit threshold sensitivity

The table below returns the top care-unit contributors at each threshold. A unit that remains near the top across thresholds is a more robust long-tail priority than one whose rank depends entirely on a single cutoff.


In [ ]:
sql_top_units_by_threshold = f"""
{TRANSFER_COHORT_CTE},
thresholds AS (
    SELECT threshold_hours
    FROM UNNEST([{threshold_sql_values}]) AS threshold_hours
),
threshold_careunit_rollup AS (
    SELECT
        threshold_hours,
        careunit,
        COUNTIF(hours_in_careunit > threshold_hours)
            AS long_segment_n,
        COUNT(DISTINCT IF(
            hours_in_careunit > threshold_hours,
            subject_id,
            NULL
        )) AS long_patient_n,
        SUM(GREATEST(hours_in_careunit - threshold_hours, 0))
            AS excess_hours
    FROM transfer_cohort
    CROSS JOIN thresholds
    GROUP BY threshold_hours, careunit
),
publishable AS (
    SELECT
        threshold_hours,
        careunit,
        long_segment_n,
        long_patient_n,
        ROUND(excess_hours, 2) AS excess_hours
    FROM threshold_careunit_rollup
    WHERE long_segment_n >= {MIN_CELL_N}
      AND long_patient_n >= {MIN_CELL_N}
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY threshold_hours
            ORDER BY excess_hours DESC
        ) AS burden_rank
    FROM publishable
)
SELECT
    threshold_hours,
    burden_rank,
    careunit,
    long_segment_n,
    long_patient_n,
    excess_hours
FROM ranked
WHERE burden_rank <= 10
ORDER BY threshold_hours, burden_rank
"""

top_units_by_threshold_df = run_query(
    sql_top_units_by_threshold,
    "top-units-by-threshold",
)
safe_display(top_units_by_threshold_df, max_rows=50)


## 12. Automated interpretation

This summary is generated only from aggregate query results.


In [ ]:
primary = primary_burden_df.iloc[0]
top_unit = (
    careunit_burden_df.iloc[0]
    if not careunit_burden_df.empty
    else None
)
top_group = (
    broad_group_burden_df.iloc[0]
    if not broad_group_burden_df.empty
    else None
)

ed_matches = cohort_component_burden_df.loc[
    cohort_component_burden_df["cohort_component"] == "ED-only"
]
ed_row = ed_matches.iloc[0] if not ed_matches.empty else None

units_to_80 = first_rank_reaching(
    careunit_burden_df,
    "cumulative_pct_of_all_excess_hours",
    80,
)

summary_lines = [
    "## Long-stay care-unit burden summary",
    "",
    f"- The analytical cohort contains **{int(primary['segment_n']):,}** "
    "valid transfer segments and retains ED-only rows with `hadm_id IS NULL`.",
    f"- **{int(primary['long_segment_n']):,}** segments "
    f"(**{float(primary['pct_segments_long']):.1f}%**) last longer than "
    f"{PRIMARY_THRESHOLD_HOURS} hours.",
    f"- Those long segments account for "
    f"**{float(primary['pct_total_hours_in_long_segments']):.1f}%** "
    "of all recorded segment-hours.",
    f"- Total time accumulated after the first "
    f"{PRIMARY_THRESHOLD_HOURS} hours is "
    f"**{format_hours(float(primary['excess_hours_beyond_72h']))}**.",
]

if top_unit is not None:
    summary_lines.append(
        f"- **{top_unit['careunit']}** is the largest visible care-unit "
        f"contributor, with "
        f"**{format_hours(float(top_unit['excess_hours_beyond_72h']))}** "
        "beyond 72 hours."
    )

if top_group is not None:
    summary_lines.append(
        f"- The largest broad group is "
        f"**{top_group['broad_careunit_group']}**, accounting for "
        f"**{float(top_group['pct_of_all_excess_hours']):.1f}%** "
        "of system-wide excess hours."
    )

if units_to_80 is not None:
    summary_lines.append(
        f"- The first **{units_to_80}** publishable care units account for "
        "at least **80%** of system-wide excess hours."
    )

if ed_row is not None:
    summary_lines.append(
        f"- ED-only encounters contribute "
        f"**{int(ed_row['long_segment_n']):,}** segments above 72 hours "
        f"and **{format_hours(float(ed_row['excess_hours_beyond_72h']))}** "
        "of excess time."
    )

summary_lines.extend([
    "",
    "**Interpretation:** these results locate and size the long-duration "
    "burden. They do not identify medical readiness, avoidable delay, "
    "or causal bottlenecks.",
])

display(Markdown("\n".join(summary_lines)))


## 13. Interpretation and limitations

This notebook identifies where long-duration burden is located. It does **not** establish:

- when a patient became medically ready for transfer or discharge;
- which hours were avoidable;
- the cause of a long segment;
- whether a care unit is inefficient;
- the causal effect of downstream-care constraints;
- whether a particular patient should have left sooner.

Long duration may reflect appropriate clinical care, case complexity, transfer practices, documentation practices, payer processes, transport, or downstream capacity.

The next operational step is not to label all excess hours as waste. It is to investigate the highest-burden units using structured timestamps for medical readiness, referral initiation, payer authorization, placement acceptance, downstream bed availability, transport booking, and actual departure.


**Citations**

- Johnson, A. et al. *MIMIC-IV (version 3.1).* PhysioNet (2024). https://doi.org/10.13026/kpb9-mt58
- Johnson, A.E.W. et al. *MIMIC-IV, a freely accessible electronic health record dataset.* Scientific Data 10, 1 (2023). https://doi.org/10.1038/s41597-022-01899-x


## v1.1 - Cohort correction in this version

The system-wide cohort **includes valid ED-only segments** whose `hadm_id` is null.

The shared cohort:

- reads directly from `physionet-data.mimiciv_3_1_hosp.transfers`;
- does **not** join to `admissions`;
- does **not** require `hadm_id IS NOT NULL`;
- requires non-null `careunit`, `intime`, and `outtime`;
- requires a positive duration;
- excludes durations longer than 365 days as probable administrative artifacts.

Because Emergency Department and observation locations are included, the notebook uses **segment-hours** or **recorded care-unit hours**, not “bed-hours.”